# שבוע 05: ניתוח פרוקרוסטס (GPA)

**Generalized Procrustes Analysis** מיישרת את כל הדגימות:  
מסירה הבדלי מיקום, גודל וסיבוב — ומשאירה רק את ה**צורה**.

נשתמש ב-`morphops` ונשווה לפני ואחרי.

In [ ]:
!pip install morphops python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import morphops as mops
rtl = get_display
print('הכל מוכן!')

In [ ]:
def parse_tps(filepath_or_text):
    """Works with both local path string and text content."""
    if '\n' in filepath_or_text:
        lines = filepath_or_text.strip().split('\n')
    else:
        with open(filepath_or_text, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    specimens, ids = [], []
    i = 0
    while i < len(lines):
        line = lines[i].strip() if hasattr(lines[i], 'strip') else lines[i]
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1].strip())
        i += 1
    return np.array(specimens), ids

print('parse_tps מוכן')

## טעינת שני קבצי TPS

In [ ]:
import urllib.request

BASE = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'

def load_tps_url(url):
    with urllib.request.urlopen(url, timeout=15) as r:
        return r.read().decode('utf-8', errors='replace')

try:
    had_text = load_tps_url(BASE + 'hadrian.tps')
    ant_text = load_tps_url(BASE + 'antoninus.tps')
    had_lm, had_ids = parse_tps(had_text)
    ant_lm, ant_ids = parse_tps(ant_text)
    print(f'הדריאנוס: {len(had_lm)} מטבעות, {had_lm.shape[1]} נקודות ציון')
    print(f'אנטונינוס פיוס: {len(ant_lm)} מטבעות, {ant_lm.shape[1]} נקודות ציון')
    DATA_OK = True
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    n_lm = 16
    angles = np.linspace(0, 2*np.pi, n_lm, endpoint=False)
    base = np.column_stack([np.cos(angles)*100, np.sin(angles)*80])
    had_lm = np.array([base + np.random.randn(n_lm, 2)*5 for _ in range(22)])
    ant_lm = np.array([base * 0.9 + np.random.randn(n_lm, 2)*5 + [10, 5] for _ in range(15)])
    had_ids = [f'Hadrian_{i+1}' for i in range(22)]
    ant_ids = [f'Antoninus_{i+1}' for i in range(15)]
    DATA_OK = False

all_lm = np.concatenate([had_lm, ant_lm], axis=0)
labels = np.array(['Hadrian']*len(had_lm) + ['Antoninus']*len(ant_lm))
print(f'\nסה"כ: {len(all_lm)} מטבעות, תוויות: {np.unique(labels, return_counts=True)}')

## הרצת GPA

`mops.gpa()` מחזירה מילון עם המפתחות `'aligned'` ו-`'mean'`.  
**חשוב**: אל תשתמשו ב-`mops.procrustes()` — ה-API הנכון הוא `mops.gpa()`.

In [ ]:
result = mops.gpa(all_lm)
aligned = result['aligned']
mean_shape = result['mean']

print(f'צורת מערך המיושר: {aligned.shape}')
print(f'צורת הממוצע: {mean_shape.shape}')
print('GPA הושלם בהצלחה!')

## גרף לפני ואחרי GPA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# לפני
ax = axes[0]
for i, lm in enumerate(all_lm):
    clr = 'steelblue' if labels[i] == 'Hadrian' else 'coral'
    ax.scatter(lm[:, 0], lm[:, 1], color=clr, s=8, alpha=0.5)
ax.set_title(rtl('לפני GPA'), fontsize=13)
ax.set_aspect('equal')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# אחרי
ax = axes[1]
for i, lm in enumerate(aligned):
    clr = 'steelblue' if labels[i] == 'Hadrian' else 'coral'
    ax.scatter(lm[:, 0], lm[:, 1], color=clr, s=8, alpha=0.5)
ax.scatter(mean_shape[:, 0], mean_shape[:, 1],
           color='black', s=40, zorder=5, label=rtl('צורה ממוצעת'))
ax.set_title(rtl('אחרי GPA'), fontsize=13)
ax.set_aspect('equal')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend(fontsize=9)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', label='Hadrian', markersize=8),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='coral', label='Antoninus', markersize=8)
]
axes[1].legend(handles=legend_elements, loc='upper right')

plt.suptitle(rtl('השפעת GPA על נקודות הציון'), fontsize=14)
plt.tight_layout()
plt.show()

## גודל קנטרואיד: השוואה בין הקבוצות

In [ ]:
def centroid_size(lm):
    c = lm.mean(axis=0)
    return np.sqrt(np.sum((lm - c)**2))

cs_had = np.array([centroid_size(s) for s in had_lm])
cs_ant = np.array([centroid_size(s) for s in ant_lm])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cs_had, bins=8, color='steelblue', alpha=0.7,
        label=f'Hadrian (n={len(cs_had)})')
ax.hist(cs_ant, bins=8, color='coral', alpha=0.7,
        label=f'Antoninus (n={len(cs_ant)})')
ax.axvline(cs_had.mean(), color='steelblue', lw=2, linestyle='--')
ax.axvline(cs_ant.mean(), color='coral', lw=2, linestyle='--')
ax.set_xlabel(rtl('גודל קנטרואיד'), fontsize=11)
ax.set_ylabel(rtl('תדירות'), fontsize=11)
ax.set_title(rtl('גודל קנטרואיד: הדריאנוס vs אנטונינוס פיוס'), fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'הדריאנוס — ממוצע: {cs_had.mean():.2f}, SD: {cs_had.std():.2f}')
print(f'אנטונינוס — ממוצע: {cs_ant.mean():.2f}, SD: {cs_ant.std():.2f}')

## סיכום

- **GPA** משתמש ב-`mops.gpa(all_lm)` ומחזיר `{'aligned': ..., 'mean': ...}`
- לאחר GPA, כל הדגימות קטנות לאותו גודל ומיושרות לציר משותף
- גודל הקנטרואיד נשמר **לפני** GPA — הוא מידע חשוב על גודל פיזי

**שאלות לחשיבה:**
1. מה מייצגת 'הצורה הממוצעת' שהגרף מציג?
2. מה יקרה אם נריץ GPA רק על הדריאנוס?
3. האם גדלי הקנטרואיד שונים בין הקיסרים? מה המשמעות הארכיאולוגית?